# Libraries

In [1]:
import subprocess
import os
import shutil
import sqlite3
import datetime as dt
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import MathsUtilities as MUte
from matplotlib.gridspec import GridSpec
from matplotlib.lines import Line2D
from matplotlib.ticker import MultipleLocator
from pathlib import Path
from IPython.display import HTML

%matplotlib inline

def ScrollTable(df,include_index = False):
    return HTML(f'''
    <div style="max-height: 300px; overflow: auto; border: 1px solid #ccc; padding: 4px;">
      {df.to_html(index=include_index)}
    </div>
    ''')

# Constants

In [2]:
Colors = {1:'#000000',
2:'#E69F00',
3:'#56B4E9',
4:'#009E73',
5:'#F0E442',
6:'#0072B2',
7:'#D55E00',
8:'#CC79A7',
9:'#1F77B4',
10:'#AEC7E8',
11:'#FF7F0E',
12:'#FFBB78',
13:'#2CA02C',
14:'#98DF8A',
15:'#D62728',
16:'#FF9896',
17:'#9467BD',
18:'#C5B0D5',
19:'#8C564B',
20:'#C49C94',
21:'#E377C2',
22:'#F7B6D2',
23:'#7F7F7F',
24:'#C7C7C7',
25:'#BCBD22',
26:'#DBDB8D',
27:'#17BECF',
28:'#9EDAE5'}

Markers = {1: 'o',
 2: '^',
 3: 's',
 4: '*',
 5: '>',
 6: 'v',
 7: '+',
 8: 'X',
 9: '<',
 10: 'p',
 11: '8',
 12: 'd',
 13:'P',
 14:'D',
 15:'o',
 16:'^'}

Lines = {1: '-',
 2: '--',
 3: '-,',
 4: ':',
 5: '-',
 6: '--',
 7: '-,',
 8: ':',
 9: '-',
 10: '--',
 11: '-,',
 12: ':',
 13: '-',
 14: '--',
 15: '-,',
 16: ':'}

branches = {"master":"master",#"master", 
            "dev":"WinterCerealWheatRelease", 
            "working":"FittingWheat"}   # branches to test
            
full_file_paths = [
    r'C:\GitHubRepos\ApsimX\Tests\Validation\Wheat\GxExM\GxExM.apsimx',
    r'C:\GitHubRepos\ApsimX\Tests\Validation\Wheat\Wheat.apsimx',
    r'C:\GitHubRepos\ApsimX\Tests\Validation\Wheat\FAR\FAR.apsimx',
    r'C:\GitHubRepos\ApsimX\Tests\Validation\Wheat\Pask\PaskExperiments.apsimx',
    r'C:\GitHubRepos\ApsimX\Tests\Validation\Wheat\Dookie2024\Dookie2024.apsimx',
]

file_names = [os.path.basename(s.rstrip("\\/")) for s in full_file_paths]

files = dict(zip(file_names , full_file_paths))

apsim_exe = r"C:\GitHubRepos\ApsimX\bin\Release\net8.0\Models.exe"  # adjust path to APSIM executable
msbuild_path = r"C:\Program Files\Microsoft Visual Studio\2022\Professional\MSBuild\Current\Bin\MSBuild.exe"
apsimSolution_path = r"C:\GitHubRepos\ApsimX\ApsimX.sln"
repo_path = r"C:\GitHubRepos\ApsimX"

In [3]:
files

{'GxExM.apsimx': 'C:\\GitHubRepos\\ApsimX\\Tests\\Validation\\Wheat\\GxExM\\GxExM.apsimx',
 'Wheat.apsimx': 'C:\\GitHubRepos\\ApsimX\\Tests\\Validation\\Wheat\\Wheat.apsimx',
 'FAR.apsimx': 'C:\\GitHubRepos\\ApsimX\\Tests\\Validation\\Wheat\\FAR\\FAR.apsimx',
 'PaskExperiments.apsimx': 'C:\\GitHubRepos\\ApsimX\\Tests\\Validation\\Wheat\\Pask\\PaskExperiments.apsimx',
 'Dookie2024.apsimx': 'C:\\GitHubRepos\\ApsimX\\Tests\\Validation\\Wheat\\Dookie2024\\Dookie2024.apsimx'}

# Set Branch and Run Simulation sets - Functions declaired

In [4]:
apsim_exe = r"C:\GitHubRepos\ApsimX\bin\Release\net8.0\Models.exe"  # adjust path to APSIM executable
msbuild_path = r"C:\Program Files\Microsoft Visual Studio\2022\Professional\MSBuild\Current\Bin\MSBuild.exe"
apsimSolution_path = r"C:\GitHubRepos\ApsimX\ApsimX.sln"
repo_path = r"C:\GitHubRepos\ApsimX"

def set_branch_and_build(branch):
    try:
        # Checkout branch
        subprocess.run(
            ["git", "checkout", branches[branch]],
            cwd=repo_path,   # ensure you’re in the repo
            check=True
        )
    except subprocess.CalledProcessError as e:
        raise RuntimeError(f"Git checkout failed for branch {branch}: {e}")

    try:
        # Force rebuild to avoid cached binaries
        subprocess.run(
            ["dotnet", "build", apsimSolution_path, "-c", "Release"],
            cwd=repo_path,
            check=True
        )
    except subprocess.CalledProcessError as e:
        raise RuntimeError(f"Build failed for branch {branch}: {e}")
    
def run_branch(branch, sim_file):
    print(f"\n=== Running {sim_file} on branch: {branch} ===")

    db_file = os.path.splitext(sim_file)[0] + ".db"
    branch_db_file = os.path.splitext(sim_file)[0] + f"_{branch}.db"

    if os.path.exists(db_file):
        os.remove(db_file)
        print(f"Deleted old {db_file}")

    # Pass relative path to APSIM
    rel_sim_file = os.path.relpath(sim_file, repo_path)

    result = subprocess.run(
        [apsim_exe, rel_sim_file],
        cwd=repo_path,
        env=os.environ,
        capture_output=True,
        text=True
    )

    print("STDOUT:", result.stdout)
    print("STDERR:", result.stderr)

    if result.returncode != 0:
        raise RuntimeError(f"Simulation failed for {branch}: {result.stderr}")

    shutil.copyfile(db_file, branch_db_file)
    print(f"Saved results to {branch_db_file}")

# Run all branches

In [5]:
# for branch in branches.keys():
#     set_branch_and_build(branch)
#     for file in files.values():
#         run_branch(branch, file)

# Run Working branch

In [11]:
branch = "working"
set_branch_and_build(branch)
for file in files.values():
    run_branch(branch, file)


=== Running C:\GitHubRepos\ApsimX\Tests\Validation\Wheat\GxExM\GxExM.apsimx on branch: working ===
Deleted old C:\GitHubRepos\ApsimX\Tests\Validation\Wheat\GxExM\GxExM.db
STDOUT: ----------------------------------------------
ERROR in file: Tests\Validation\Wheat\GxExM\GxExM.apsimx
Simulation name: Gatton2015CvHartog
Models.Core.SimulationException
 ---> System.Exception: Error while locating variable '[AboveGround].Wt' in variable reference '.Gatton2015CvHartog.Field.Wheat.Stem.PotentialGrowth.TargetWt.PotentialAboveGroundWt.AboveGroundWt'
 ---> System.Exception: While locating model .Wt: unknown model or property specification Wt
   at APSIM.Core.Locator.GetInternal(Node relativeTo, String namePath, LocatorFlags flags) in C:\GitHubRepos\ApsimX\APSIM.Core\Locate\Locator.cs:line 292
   at APSIM.Core.Locator.GetObject(Node relativeTo, String namePath, LocatorFlags flags) in C:\GitHubRepos\ApsimX\APSIM.Core\Locate\Locator.cs:line 61
   at APSIM.Core.Node.GetObject(String path, LocatorFl

RuntimeError: Simulation failed for working: 

# Get Harvest predictions and observations

## Function to read apsim data base

In [ ]:
import sqlite3
import pandas as pd
from pathlib import Path
from typing import Dict, Optional

def read_table_typed(
    db_path: str,
    table_name: str,
    schema: Dict[str, str],
    errors: str = "raise",
    datetime_formats: Optional[Dict[str, Optional[str]]] = None,
) -> pd.DataFrame:
    """
    Read selected columns from a SQLite table with explicit, enforced dtypes.
    If requested columns are missing, add them as empty columns with correct dtype.

    Supported dtypes: int, float, string, datetime, date, time
    Timezone-neutral: datetime columns are returned as tz-naive pandas datetime64[ns].
    """

    # --- type aliases (bool/category removed) ---
    TYPE_ALIASES = {
        "int": "int", "integer": "int", "int64": "int", "long": "int",
        "float": "float", "real": "float", "double": "float", "number": "float",
        "string": "string", "str": "string", "text": "string",
        "datetime": "datetime", "timestamp": "datetime",
        "date": "date",
        "time": "time",
    }

    def norm_dtype(dt: str) -> str:
        key = dt.strip().lower()
        if key not in TYPE_ALIASES:
            raise ValueError(
                f"Unsupported dtype '{dt}'. Supported: {sorted(set(TYPE_ALIASES.values()))}"
            )
        return TYPE_ALIASES[key]

    # --- casters (bool/category removed) ---
    def cast_int(s: pd.Series) -> pd.Series:
        out = pd.to_numeric(s, errors=("coerce" if errors == "coerce" else "raise"))
        return out.astype("Int64")  # nullable integer

    def cast_float(s: pd.Series) -> pd.Series:
        out = pd.to_numeric(s, errors=("coerce" if errors == "coerce" else "raise")).astype("float64")
        return out

    def cast_string(s: pd.Series) -> pd.Series:
        return s.astype("string")

    def cast_datetime(s: pd.Series, fmt: Optional[str]) -> pd.Series:
        # Parse as tz-naive; if input contains tz-aware values, strip the tz.
        out = pd.to_datetime(s, errors=("coerce" if errors == "coerce" else "raise"), format=fmt, utc=False)
        # If any timezone slipped in, make it tz-naive
        if hasattr(out, "dt"):
            try:
                # If tz-aware, this will remove tz info; if tz-naive, it's a no-op
                out = out.dt.tz_localize(None)
            except Exception:
                # If tz_localize(None) fails (e.g., already tz-naive), ignore
                pass
        return out

    def cast_date(s: pd.Series, fmt: Optional[str]) -> pd.Series:
        dt = pd.to_datetime(s, errors=("coerce" if errors == "coerce" else "raise"), format=fmt, utc=False)
        return dt.dt.date

    def cast_time(s: pd.Series, fmt: Optional[str]) -> pd.Series:
        dt = pd.to_datetime(s, errors=("coerce" if errors == "coerce" else "raise"), format=fmt, utc=False)
        return dt.dt.time

    CASTERS = {
        "int":      lambda col, series: cast_int(series),
        "float":    lambda col, series: cast_float(series),
        "string":   lambda col, series: cast_string(series),
        "datetime": lambda col, series: cast_datetime(series, (datetime_formats or {}).get(col)),
        "date":     lambda col, series: cast_date(series, (datetime_formats or {}).get(col)),
        "time":     lambda col, series: cast_time(series, (datetime_formats or {}).get(col)),
    }

    # --- open DB & gather available columns ---
    db_file = Path(db_path)
    if not db_file.exists():
        raise FileNotFoundError(f"Database file not found: {db_file}")

    with sqlite3.connect(db_file) as conn:
        conn.row_factory = sqlite3.Row
        cur = conn.cursor()

        cur.execute("SELECT name FROM sqlite_master WHERE type='table' AND name = ?;", (table_name,))
        if cur.fetchone() is None:
            raise ValueError(f"Table '{table_name}' not found in {db_file}")

        cur.execute(f'PRAGMA table_info("{table_name}");')
        table_cols = [r[1] for r in cur.fetchall()]  # r[1] = column name

        requested_cols = list(schema.keys())
        present_cols = [c for c in requested_cols if c in table_cols]
        missing_cols = [c for c in requested_cols if c not in table_cols]

        # Normalize requested dtypes once (even for missing columns)
        normalized_schema = {col: norm_dtype(dtype) for col, dtype in schema.items()}

        # Read only present columns; if none present, read * to get row count
        select_cols_sql = ", ".join(f'"{c}"' for c in present_cols) if present_cols else "*"
        df = pd.read_sql_query(f'SELECT {select_cols_sql} FROM "{table_name}";', conn)

    # --- cast present columns ---
    for col in present_cols:
        target = normalized_schema[col]
        caster = CASTERS.get(target)
        if caster is None:
            raise ValueError(f"Unknown normalized dtype: {target}")
        df[col] = caster(col, df[col])

    # --- add missing columns as empty and cast ---
    for col in missing_cols:
        target = normalized_schema[col]
        caster = CASTERS.get(target)
        if caster is None:
            raise ValueError(f"Unknown normalized dtype: {target}")
        empty_series = pd.Series([pd.NA] * len(df))
        df[col] = caster(col, empty_series)

    # --- preserve requested order ---
    df = df[requested_cols]
    return df

## Read in Simulation name table

In [ ]:
#Set up simulations index table
def GetSimulationData(files, branch):
    schema = {"ID": "int",
              "Name": "string"}
    allSimulations = {}
    for name, path in files.items():
        branch_db_file = os.path.splitext(path)[0] + f"_{branch}.db"
        Simulations = read_table_typed(db_path=branch_db_file,
                                       table_name="_Simulations",
                                       schema=schema,
                                       errors="raise",              # or "coerce"
                                       datetime_formats={"RunDate": "%Y-%m-%d","CreatedAt": None}
                                       ) 
        
        Simulations.set_index('ID', inplace=True)
        Simulations.sort_index(axis=0, inplace=True)
        allSimulations[name] = Simulations

    if allSimulations:
        return pd.concat(allSimulations.values(),
                         keys=allSimulations.keys(),
                         names=['SimulationFile', 'SimulationID'])
    else:
        return pd.DataFrame()

SimulationData = {}
for branch in branches.keys():
    SimulationData[branch] = GetSimulationData(files,branch)

## Read in harvest predictions at harvest time

In [ ]:
#Set up simulations index table
def GetHarvestPredictions(files, branch):
    schema = {
                "CheckpointID" : "int",
                "SimulationID" : "int",
                "IWeather.Latitude" : "float",
                "IWeather.Longitude" : "float",
                "Wheat.SowingData.Cultivar" : "string",
                "Experiment" : "string",
                'Wheat.AboveGround.N' : "float",
                'Wheat.AboveGround.NConc' : "float",
                'Wheat.AboveGround.Wt' : "float",
                'Wheat.Ear.N' : "float",
                'Wheat.Ear.NConc' : "float",
                'Wheat.Ear.Wt' : "float",
                'Wheat.Grain.N' : "float",
                'Wheat.Grain.NConc' : "float",
                'Wheat.Grain.Number' : "float",
                'Wheat.Grain.Protein' : "float",
                'Wheat.Grain.Size' : "float",
                'Wheat.Grain.Wt' : "float",
                'Wheat.Leaf.N' : "float",
                'Wheat.Leaf.StemPopulation' : "float",
                'Wheat.Leaf.Wt' : "float",
                'Wheat.Phenology.CAMP.TSHS' : "float",
                'Wheat.Phenology.CurrentStageName' : "string",
                'Wheat.Phenology.EmergenceDAS' : "float",
                'Wheat.Phenology.FinalLeafNumber' : "float",
                'Wheat.Phenology.FlagLeafDAS' : "float",
                'Wheat.Phenology.FloweringDAS' : "float",
                'Wheat.Phenology.HeadingDAS' : "float",
                'Wheat.Phenology.MaturityDAS' : "float",
                'Wheat.Phenology.TerminalSpikeletDAS' : "float",
                'Wheat.Spike.HeadNumber' : "float",
                'Wheat.Spike.N' : "float",
                'Wheat.Spike.Wt' : "float",
                'Wheat.Stem.N' : "float",
                'Wheat.Stem.NConc' : "float",
                'Wheat.Stem.Wt' : "float"
                }

    allHarvPreds = {}
    for name, path in files.items():
        branch_db_file = os.path.splitext(path)[0] + f"_{branch}.db"
        HarvPreds = read_table_typed(db_path=branch_db_file,
                                       table_name="HarvestReport",
                                       schema=schema,
                                       errors="raise",              # or "coerce"
                                       datetime_formats={"RunDate": "%Y-%m-%d","CreatedAt": None}
                                       ) 
        
        HarvPreds.set_index('SimulationID', inplace=True)
        HarvPreds.sort_index(axis=0, inplace=True)
        HarvPreds['SimulationName'] = [SimulationData[branch].loc[(name,x),'Name'] for x in HarvPreds.index]
        HarvPreds.set_index('SimulationName', inplace=True, drop=False)
        HarvPreds.sort_index(axis=0, inplace=True)
        allHarvPreds[name] = HarvPreds

    if allHarvPreds:
        return pd.concat(allHarvPreds.values(),
                         keys=allHarvPreds.keys(),
                         names=['SimulationFile', 'SimulationID'])
    else:
        return pd.DataFrame()

HarvestPredictions = {}
for branch in branches.keys():
    HarvestPredictions[branch] = GetHarvestPredictions(files,branch)

## Read in Observations

In [ ]:
#Set up simulations index table
def GetObservations(files, branch):
    schema = {
                "CheckpointID" : "int",
                "SimulationID" : "int",
                'Wheat.AboveGround.N' : "float",
                'Wheat.AboveGround.NConc' : "float",
                'Wheat.AboveGround.Wt' : "float",
                'Wheat.Ear.N' : "float",
                'Wheat.Ear.NConc' : "float",
                'Wheat.Ear.Wt' : "float",
                'Wheat.Grain.N' : "float",
                'Wheat.Grain.NConc' : "float",
                'Wheat.Grain.Number' : "float",
                'Wheat.Grain.Protein' : "float",
                'Wheat.Grain.Size' : "float",
                'Wheat.Grain.Wt' : "float",
                'Wheat.Leaf.N' : "float",
                'Wheat.Leaf.StemPopulation' : "float",
                'Wheat.Leaf.Wt' : "float",
                'Wheat.Phenology.CAMP.TSHS' : "float",
                'Wheat.Phenology.CurrentStageName' : "string",
                'Wheat.Phenology.EmergenceDAS' : "float",
                'Wheat.Phenology.FinalLeafNumber' : "float",
                'Wheat.Phenology.FlagLeafDAS' : "float",
                'Wheat.Phenology.FloweringDAS' : "float",
                'Wheat.Phenology.HeadingDAS' : "float",
                'Wheat.Phenology.MaturityDAS' : "float",
                'Wheat.Phenology.TerminalSpikeletDAS' : "float",
                'Wheat.Spike.HeadNumber' : "float",
                'Wheat.Spike.N' : "float",
                'Wheat.Spike.Wt' : "float",
                'Wheat.Stem.N' : "float",
                'Wheat.Stem.NConc' : "float",
                'Wheat.Stem.Wt' : "float"
                }

    allObs = {}
    for name, path in files.items():
        branch_db_file = os.path.splitext(path)[0] + f"_{branch}.db"
        Obs = read_table_typed(db_path=branch_db_file,
                                       table_name="Observed",
                                       schema=schema,
                                       errors="coerce",              # or "coerce"
                                       datetime_formats={"RunDate": "%Y-%m-%d","CreatedAt": None}
                                       ) 
        Obs.set_index('SimulationID', inplace=True, drop=False)
        Obs.sort_index(axis=0, inplace=True)
        Obs['SimulationName'] = [SimulationData[branch].loc[(name,x),'Name'] for x in Obs.index]
        Obs.set_index('SimulationName', inplace=True, drop=False)
        Obs.sort_index(axis=0, inplace=True)
        allObs[name] = Obs

    if allObs:
        return pd.concat(allObs.values(),
                         keys=allObs.keys(),
                         names=['SimulationFile', 'SimulationID'])
    else:
        return pd.DataFrame()

Observations = {}
for branch in branches.keys():
    Observations[branch] = GetObservations(files,branch)

## Slice out observations at harvest time

In [ ]:
#Make table of Observations tagged with a 'Wheat.Phenology.CurrentStageName' value equal to 'HarvestRipe' which indicates it is the final harvest value
HarvestObs = {}
for branch in branches.keys():
    HarvestObs[branch] = Observations[branch].loc[Observations[branch].loc[:,'Wheat.Phenology.CurrentStageName']=='HarvestRipe',:].dropna(axis=1,how='all').dropna(axis=0,how='all')
    HarvestObs[branch] = HarvestObs[branch].sort_index(axis=0)

In [ ]:
ScrollTable(HarvestPredictions['master'],include_index=True)

## Make Harvest Obs Pred graphs for all variables - Function defined

In [ ]:
def harvestObsPredGraph():
    HarvMar = {1:'o',2:'x'}
    checkObsPred = pd.DataFrame()
    legpos = {"master":[0.01,0.89],"dev":[0.6,0.2],"working":[.6,0.4]}
    HarvGraphs =  plt.figure(figsize=(20,20))
    pos=1
    obsPred = pd.DataFrame()
    for var in HarvVars:
        obsPred = pd.DataFrame()
        ax = HarvGraphs.add_subplot(6,6,pos)
        axmax = 0.01
        cpos = 1
        msv = 3
        alpv = 1

        for branch in branches:
            for file in files.keys():
                try:
                    obsPred = pd.DataFrame(HarvestObs[branch].loc[:,var].dropna().copy())
                    obsPred.columns = ['obs']
                    obsPred.obs = pd.to_numeric(obsPred.obs)
                    for s in obsPred.index:
                        try:
                            obsPred.loc[s,'pred'] = pd.to_numeric(HarvestPredictions[branch].loc[s,var].values[0])
                        except:
                            do = 'nothing'
                except:
                    do = 'nothing'

                obsPred.dropna(inplace=True)

                if not obsPred.empty:
                    ax.plot(obsPred.obs, obsPred.pred,
                            HarvMar.get(cpos,'o'),
                            ms=msv, color=Colors[cpos],
                            alpha=alpv, markeredgewidth=1)

            # Regression stats
            if not obsPred.empty and obsPred.pred.dropna().size > 2:
                n = len(obsPred.obs)
                RegStats = MUte.MathUtilities.CalcRegressionStats('',obsPred.pred,obsPred.obs)
                fitR2 = f"{branch}\n$NSE$ = {RegStats.NSE:.3f}\nn = {n}"
                ax.text(legpos[branch][0], legpos[branch][1], fitR2,
                        transform=ax.transAxes, ha='left', va='top', color=Colors[cpos])

            if not obsPred.empty:
                axmax = max(obsPred.obs.max(), obsPred.pred.max(), axmax)

            cpos += 1
            alpv *= 0.9

        ax.text(0.01,0.99,var,transform=ax.transAxes,ha='left',va='top')
        ax.set_ylim(0, axmax*1.05)
        ax.set_xlim(0, axmax*1.05)
        pos += 1

In [ ]:
HarvVars = ['Wheat.AboveGround.N',
'Wheat.AboveGround.NConc',
'Wheat.AboveGround.Wt',
'Wheat.Ear.N',
'Wheat.Ear.NConc',
'Wheat.Ear.Wt',
'Wheat.Grain.N',
'Wheat.Grain.NConc',
'Wheat.Grain.Number',
'Wheat.Grain.Protein',
'Wheat.Grain.Size',
'Wheat.Grain.Wt',
'Wheat.Leaf.N',
'Wheat.Leaf.StemPopulation',
'Wheat.Leaf.Wt',
'Wheat.Phenology.CAMP.TSHS',
'Wheat.Phenology.EmergenceDAS',
'Wheat.Phenology.FinalLeafNumber',
'Wheat.Phenology.FlagLeafDAS',
'Wheat.Phenology.FloweringDAS',
'Wheat.Phenology.HeadingDAS',
'Wheat.Phenology.MaturityDAS',
'Wheat.Phenology.TerminalSpikeletDAS',
'Wheat.Spike.HeadNumber',
'Wheat.Spike.N',
'Wheat.Spike.Wt',
'Wheat.Stem.N',
'Wheat.Stem.NConc',
'Wheat.Stem.Wt']

# Harvest Obs Pred - Render Graphs

In [ ]:
harvestObsPredGraph()

# Get daily observations and predictions

## Define funciton to get all data for a single variable

In [ ]:
def getVariableData(var, branch, file_name, file_path):
    schema = {'SimulationID' : "int",
              'Clock.Today' : "date",
              "Experiment" : "string",
              'Wheat.Phenology.Stage' : "float",
              var : "float"
              }
    branch_db_file = os.path.splitext(file_path)[0] + f"_{branch}.db"
    Pred = read_table_typed(db_path=branch_db_file,
                                   table_name="DailyReport",
                                   schema=schema,
                                   errors="coerce",              # or "coerce"
                                   datetime_formats={"RunDate": "%Y-%m-%d","CreatedAt": None}
                                   )
    Pred_clean = Pred[Pred["Wheat.Phenology.Stage"] >= 2]
    Obs = read_table_typed(db_path=branch_db_file,
                                   table_name="Observed",
                                   schema=schema,
                                   errors="coerce",              # or "coerce"
                                   datetime_formats={"RunDate": "%Y-%m-%d","CreatedAt": None}
                                   )
    Obs_clean = (
                  Obs.dropna(subset=[var])
                 .drop_duplicates(subset=['SimulationID', 'Clock.Today'], keep='last')
                 .rename(columns={var: var+'_Obs'})
                 )
    obsPred = (
                Pred_clean.merge(
                           Obs_clean[['SimulationID', 'Clock.Today', var+'_Obs']],
                           on=['SimulationID', 'Clock.Today'],
                           how='left'
                           )
               )

    obsPred['Residuals'] = obsPred[var+'_Obs']-obsPred[var]
    
    name_map = (SimulationData[branch].xs(file_name, level='SimulationFile')['Name'])
    obsPred['SimulationName'] = obsPred['SimulationID'].map(name_map)
    return obsPred

## Function to make graph of residules over the crops development stage

In [ ]:
def residualGraph(var,Var_obsPred):
    fig = plt.figure(figsize=(15, 6))
    pos = 1
    ymax = 0
    for branch in branches:
        for file_name in files.keys():
            ymax = max(ymax,abs(Var_obsPred[branch][file_name]['Residuals']).max())
    for branch in branches:
        ax = fig.add_subplot(1, 3, pos)
        colorPos = 1
        markerPos = 1
        for file_name in files.keys():
            expts = list(Var_obsPred[branch][file_name]['Experiment'].drop_duplicates().values)
            for exp in expts:
                expFilter = Var_obsPred[branch][file_name]['Experiment'] == exp
                if Var_obsPred[branch][file_name].loc[expFilter,'Residuals'].count() > 0:
                    sims = list(Var_obsPred[branch][file_name].loc[expFilter,'SimulationID'].drop_duplicates().values)
                    for sim in sims:
                        simFilter = Var_obsPred[branch][file_name]["SimulationID"] == sim
                        resids = Var_obsPred[branch][file_name].loc[simFilter,["Wheat.Phenology.Stage","Residuals"]].dropna()
                        plt.plot(resids["Wheat.Phenology.Stage"],
                                 resids["Residuals"],
                                 Markers[markerPos]+'-',color = Colors[colorPos],label=exp)

                    colorPos += 1
                    if colorPos > 8:
                        colorPos = 1
                        markerPos += 1

        # Regression annotation (based on obs/pred)
        allobs = []
        allpred = []
        for file_name in files.keys():
            obs  = Var_obsPred[branch][file_name][var+"_Obs"].dropna()
            allobs = allobs + list(obs)
            pred = Var_obsPred[branch][file_name][var].reindex(obs.index)
            allpred = allpred + list(pred)
        if len(allobs) > 2:
            RegStats = MUte.MathUtilities.CalcRegressionStats('', allpred, allobs)
            txt = f"{branch}\n$NSE$ = {RegStats.NSE:.3f}\nn = {len(allobs)}"
            ax.text(0.05, 1.1, txt, transform=ax.transAxes, ha='left', va='top')

        plt.ylim(-ymax,ymax)
        pos+=1

    last_ax = fig.axes[-1]
    handles, labels = last_ax.get_legend_handles_labels()

    # Deduplicate labels while preserving first occurrence
    by_label = {}
    for h, lab in zip(handles, labels):
        if lab not in by_label:
            by_label[lab] = h

    # Make room on the right for legend
    fig.subplots_adjust(right=0.82)

    last_ax.legend(
        list(by_label.values()), list(by_label.keys()),
        loc='center left',
        bbox_to_anchor=(1.02, 0.5),  # 1.02 pushes just outside the right edge of the axis
        fontsize=12, title='Experiment', frameon=True,
        borderaxespad=0.0
    )
    return fig

## Helper functions for time series graphs

In [ ]:
def getFigDimensions(Var_obsPred):
    totalcolumns = 6
    totalrows = 0
    totalPanels = 0
    branch = 'master'
    ymax = 0
    for file_name in files.keys():
        expts = list(Var_obsPred[branch][file_name]['Experiment'].drop_duplicates().values)
        for exp in expts:
            numSims = 0
            expFilter = Var_obsPred[branch][file_name]['Experiment'] == exp
            if Var_obsPred[branch][file_name].loc[expFilter,var+'_Obs'].count() > 1:
                sims = list(Var_obsPred[branch][file_name].loc[expFilter,'SimulationID'].drop_duplicates().values)
                for sim in sims:
                    simFilter = (Var_obsPred[branch][file_name]["SimulationID"] == sim)
                    if Var_obsPred[branch][file_name].loc[simFilter,var+'_Obs'].count() > 1:
                        totalPanels +=1
                        numSims +=1
                        ymax = max(ymax,Var_obsPred[branch][file_name].loc[simFilter,var+'_Obs'].max())
                        ymax = max(ymax,Var_obsPred[branch][file_name].loc[simFilter,var].max())
            totalrows += int(np.ceil(numSims / totalcolumns))
    return totalcolumns, totalrows, ymax

def isFirstInRow(n: int, x: int) -> bool:
    """
    Returns True if n == 1 or n is a multiple of x plus 1.
    Example: x=10 → True for 1, 11, 21, 31, ...
    """
    if x <= 0:
        raise ValueError("x must be > 0")
    return n == 1 or (n - 1) % x == 0

styles = {
    "master":   dict(ls="-", lw=1.3,   alpha=0.6, color = '#000000'),
    "dev":      dict(ls="--",  lw=1.3, alpha=0.8, color = '#E69F00'),
    "working":  dict(ls="-.",  lw=1.3,   alpha=1.0, color = '#56B4E9'),
        }

## Function to generate time series graph of variavle and observations over development time series

In [ ]:
def timeSeriesByTreatment(var, Var_obsPred):
    pos=1
    totalcolumns, totalrows, ymax = getFigDimensions(Var_obsPred)
    fig = plt.figure(figsize=(20, totalrows * 3))
    for file_name in files.keys():
        expts = list(Var_obsPred['master'][file_name]['Experiment'].drop_duplicates().values)
        for exp in expts:
            expFilter = Var_obsPred['master'][file_name]['Experiment'] == exp
            if Var_obsPred['master'][file_name].loc[expFilter,var+'_Obs'].count() > 1:
                sims = list(Var_obsPred['master'][file_name].loc[expFilter,'SimulationName'].drop_duplicates().values)
                for sim in sims:
                    masterFilter = (Var_obsPred['master'][file_name]["SimulationName"] == sim)
                    sortedMaster = (Var_obsPred['master'][file_name].loc[masterFilter, :].sort_values('Wheat.Phenology.Stage'))
                    if Var_obsPred['master'][file_name].loc[masterFilter,var+'_Obs'].count() > 1:
                        ax = fig.add_subplot(totalrows,totalcolumns,pos)
                        for branch in branches:
                            simFilter = (Var_obsPred[branch][file_name]["SimulationName"] == sim)
                            sortedDF = Var_obsPred[branch][file_name].loc[simFilter,:].sort_values('Wheat.Phenology.Stage')
                            ax.plot(sortedDF.loc[:,'Wheat.Phenology.Stage'],
                                     sortedDF.loc[:,var],
                                     styles[branch]["ls"], lw=styles[branch]["lw"], color = styles[branch]["color"])#, alpha=styles[branch]["lw"]['alpha'])
                        ax.plot(sortedMaster.loc[:,'Wheat.Phenology.Stage'],
                                 sortedMaster.loc[:,var+'_Obs'],
                                 'o',color='green')
                        ax.set_xlim(1, 11.5)
                        ax.xaxis.set_major_locator(MultipleLocator(1.0))
                        ax.set_xlabel("")  # no x-axis label
                        plt.ylim(0,ymax)
                        ax.set_xticklabels([])
                        if isFirstInRow(pos,totalcolumns):
                            ax.set_ylabel(var)
                            plt.text(0.05,0.9,exp,transform=ax.transAxes,fontsize=12)
                        else:
                            ax.set_ylabel("")
                            ax.set_yticklabels([])
                        plt.text(0.05,0.8,sim.replace(exp,""),transform=ax.transAxes,fontsize=10)
                        pos+=1
                if (pos - 1) % totalcolumns != 0:
                    pos = int(np.ceil(pos / totalcolumns) * totalcolumns) + 1
    return fig

## Driver function to get variable data and produce graphs

In [ ]:
def makeGraphs(var):
    try:
        var_obsPred = {}
        for branch in branches.keys():
            branch_obsPred = {}
            for name, path in files.items():
                branch_obsPred[name] = getVariableData(var, branch, name, path)
            var_obsPred[branch] = branch_obsPred
        fig1 = residualGraph(var, var_obsPred)
        
        fig2 = timeSeriesByTreatment(var, var_obsPred)
        return None  # success

    except Exception as e:
        return e

# Time series graphs for individual variables

## Leaf.LAI

In [ ]:
var = 'Wheat.Leaf.LAI'
err = makeGraphs(var)
if err is not None:
    print(f"{var} failed with error: {err}")

## Spectral.NDVI

In [ ]:
var = 'Spectral.NDVI'
err = makeGraphs(var)
if err is not None:
    print(f"{var} failed with error: {err}")

## Leaf.CoverGreen

In [ ]:
var = 'Wheat.Leaf.CoverGreen'
err = makeGraphs(var)
if err is not None:
    print(f"{var} failed with error: {err}")

## Leaf.CoverTotal

In [ ]:
var = 'Wheat.Leaf.CoverTotal'
err = makeGraphs(var)
if err is not None:
    print(f"{var} failed with error: {err}")

## Leaf.Live.Wt

In [ ]:
var = 'Wheat.Leaf.Live.Wt'
err = makeGraphs(var)
if err is not None:
    print(f"{var} failed with error: {err}")

## Leaf.Wt

In [ ]:
var = 'Wheat.Leaf.Wt'
err = makeGraphs(var)
if err is not None:
    print(f"{var} failed with error: {err}")

## Leaf.SpecificAreaCanopy

In [ ]:
var = 'Wheat.Leaf.SpecificAreaCanopy'
err = makeGraphs(var)
if err is not None:
    print(f"{var} failed with error: {err}")

## Leaf.StemNumberPerPlant

In [ ]:
var = 'Wheat.Leaf.StemNumberPerPlant'
err = makeGraphs(var)
if err is not None:
    print(f"{var} failed with error: {err}")

## Leaf.StemPopulation

In [ ]:
var = 'Wheat.Leaf.StemPopulation'
err = makeGraphs(var)
if err is not None:
    print(f"{var} failed with error: {err}")

## AboveGround.Wt

In [ ]:
var = 'Wheat.AboveGround.Wt'
err = makeGraphs(var)
if err is not None:
    print(f"{var} failed with error: {err}")

## Ear.Wt

In [ ]:
var = 'Wheat.Ear.Wt'
err = makeGraphs(var)
if err is not None:
    print(f"{var} failed with error: {err}")

## Grain.Wt

In [ ]:
var = 'Wheat.Grain.Wt'
err = makeGraphs(var)
if err is not None:
    print(f"{var} failed with error: {err}")

## Leaf.Dead.Wt

In [ ]:
var = 'Wheat.Leaf.Dead.Wt'
err = makeGraphs(var)
if err is not None:
    print(f"{var} failed with error: {err}")

## Leaf.Live.StorageWt

In [ ]:
var = 'Wheat.Leaf.Live.StorageWt'
err = makeGraphs(var)
if err is not None:
    print(f"{var} failed with error: {err}")

## Leaf.Live.Wt

In [ ]:
var = 'Wheat.Leaf.Live.Wt'
err = makeGraphs(var)
if err is not None:
    print(f"{var} failed with error: {err}")

## Spike.Live.StorageWt

In [ ]:
var = 'Wheat.Spike.Live.StorageWt'
err = makeGraphs(var)
if err is not None:
    print(f"{var} failed with error: {err}")

## Spike.Wt

In [ ]:
var = 'Wheat.Spike.Wt'
err = makeGraphs(var)
if err is not None:
    print(f"{var} failed with error: {err}")

## Stem.Live.StorageWt

In [ ]:
var = 'Wheat.Stem.Live.StorageWt'
err = makeGraphs(var)
err = makeGraphs('Wheat.Stem.Live.StorageWt')
if err is not None:
    print(f"{var} failed with error: {err}")

## Stem.Wt

In [ ]:
var = 'Wheat.Stem.Wt'
err = makeGraphs(var)
if err is not None:
    print(f"{var} failed with error: {err}")

## AboveGround.N

In [ ]:
var = 'Wheat.AboveGround.N'
err = makeGraphs(var)
if err is not None:
    print(f"{var} failed with error: {err}")

## AboveGround.NConc

In [ ]:
var = 'Wheat.AboveGround.NConc'
err = makeGraphs(var)
if err is not None:
    print(f"{var} failed with error: {err}")

## Ear.N

In [ ]:
var = 'Wheat.Ear.N'
err = makeGraphs(var)
if err is not None:
    print(f"{var} failed with error: {err}")

## Ear.NConc

In [ ]:
var = 'Wheat.Ear.NConc'
err = makeGraphs(var)
if err is not None:
    print(f"{var} failed with error: {err}")

## Grain.N

In [ ]:
var = 'Wheat.Grain.N'
err = makeGraphs(var)
if err is not None:
    print(f"{var} failed with error: {err}")

## Grain.NConc

In [ ]:
var = 'Wheat.Grain.NConc'
err = makeGraphs(var)
if err is not None:
    print(f"{var} failed with error: {err}")

## Leaf.Dead.N

In [ ]:
var = 'Wheat.Leaf.Dead.N'
err = makeGraphs(var)
if err is not None:
    print(f"{var} failed with error: {err}")

## Leaf.Dead.NConc

In [ ]:
var = 'Wheat.Leaf.Dead.NConc'
err = makeGraphs(var)
if err is not None:
    print(f"{var} failed with error: {err}")

## Leaf.Live.N

In [ ]:
var = 'Wheat.Leaf.Live.N'
err = makeGraphs(var)
if err is not None:
    print(f"{var} failed with error: {err}")

## Leaf.Live.NConc

In [ ]:
var = 'Wheat.Leaf.Live.NConc'
err = makeGraphs(var)
if err is not None:
    print(f"{var} failed with error: {err}")

## Leaf.N

In [ ]:
var = 'Wheat.Leaf.N'
err = makeGraphs(var)
if err is not None:
    print(f"{var} failed with error: {err}")

## Spike.N

In [ ]:
var = 'Wheat.Spike.N'
err = makeGraphs(var)
if err is not None:
    print(f"{var} failed with error: {err}")

## spike.NConc

In [ ]:
var = 'Wheat.Spike.NConc'
err = makeGraphs(var)
if err is not None:
    print(f"{var} failed with error: {err}")

## Stem.N

In [ ]:
var = 'Wheat.Stem.N'
err = makeGraphs(var)
if err is not None:
    print(f"{var} failed with error: {err}")

## Stem.NConc

In [ ]:
var = 'Wheat.Stem.NConc'
err = makeGraphs(var)
if err is not None:
    print(f"{var} failed with error: {err}")

## Phenology.HaunStage

In [ ]:
var = 'Wheat.Phenology.HaunStage'
err = makeGraphs(var)
if err is not None:
    print(f"{var} failed with error: {err}")

## Leaf.Height

In [ ]:
var = 'Wheat.Leaf.Height'
err = makeGraphs(var)
if err is not None:
    print(f"{var} failed with error: {err}")

## Phenology.Zadok.Stage

In [ ]:
var = 'Wheat.Phenology.Zadok.Stage'
err = makeGraphs(var)
if err is not None:
    print(f"{var} failed with error: {err}")